# 1. Exploratory Data Analysis & Feature Engineering

## 1.1. Load Data and Initial Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('ggplot')

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
df.head()

In [ ]:
df.info()

## 1.2. Data Cleaning

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.drop(columns=['customerID'], inplace=True)
median_total_charges = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_total_charges, inplace=True)
df.isnull().sum()

## 1.3. EDA

In [ ]:
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['Contract', 'InternetService', 'PaymentMethod']

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

churn_counts = df['Churn'].value_counts()
axes[0].bar(churn_counts.index, churn_counts.values)
axes[0].set_title('Churn Distribution')
axes[0].set_xticks([0, 1])

for i, feature in enumerate(numerical_features):
    ax = axes[i+1]
    df[df['Churn']==0][feature].plot(kind='hist', ax=ax, label='No Churn', alpha=0.5, density=True)
    df[df['Churn']==1][feature].plot(kind='hist', ax=ax, label='Churn', alpha=0.5, density=True)
    ax.set_title(f'{feature} by Churn')
    ax.legend()

for i, feature in enumerate(categorical_features):
    ax = axes[i+4]
    churn_by_feature = df.groupby([feature, 'Churn']).size().unstack(fill_value=0)
    churn_by_feature.plot(kind='bar', stacked=True, ax=ax)
    ax.set_title(f'{feature} vs. Churn')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 1.4. Feature Engineering

In [ ]:
df_processed = df.copy()
categorical_cols = df_processed.select_dtypes(include=['object', 'category']).columns
df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

labels = ['0-1yr', '1-2yr', '2-3yr', '3-4yr', '4-5yr', '5+yr']
df_processed['tenure_bins'] = pd.cut(df_processed['tenure'], bins=[0, 12, 24, 36, 48, 60, 72], labels=labels, right=False)
df_processed = pd.get_dummies(df_processed, columns=['tenure_bins'], drop_first=True)

df_processed.head()

## 1.5. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 12))
corr_matrix = df_processed.corr()
churn_corr = corr_matrix[['Churn']].sort_values(by='Churn', ascending=False)

cax = plt.matshow(churn_corr, cmap='coolwarm')
plt.colorbar(cax)

plt.title('Feature Correlation with Churn', pad=20)
plt.xticks(ticks=np.arange(len(churn_corr.columns)), labels=churn_corr.columns, rotation=90)
plt.yticks(ticks=np.arange(len(churn_corr.index)), labels=churn_corr.index)

for i in range(len(churn_corr.index)):
    for j in range(len(churn_corr.columns)):
        plt.text(j, i, f'{churn_corr.iloc[i, j]:.2f}', ha='center', va='center', color='black')

plt.show()

## 1.6. Save Processed Data

In [ ]:
output_path = '../data/processed/churn_processed_data.csv'
df_processed.to_csv(output_path, index=False)
print(f'Processed data saved to {output_path}')